# 第8章 励磁系统 (Excitation Systems)

> Kundur P. *Power System Stability and Control*, McGraw-Hill, 1994, Chapter 8

## 本章概述

励磁系统是同步发电机电压控制的核心，直接影响:
1. **稳态电压调节**: 维持机端电压在设定值
2. **暂态稳定**: 快速励磁响应提高第一摆稳定裕度
3. **小干扰稳定**: AVR 通过 K5 通道可能引入负阻尼

本章使用 **IEEE Type 1 (IEEET1)** 励磁模型进行演示分析。

### IEEET1 框图
```
Vref ---[+]---[KA/(1+sTA)]---[VR]---[1/(KE+sTE)]---[Efd]
        -|                        |         |
         |        [sKF/(1+sTF)]--|         |
         |                        |         |
        Vt <------------------------+---------|
         |                                    |
        Vs (from PSS)                        SE(Efd)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from psa4teaching.models import IEEET1Params

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('Import OK')

---
## 8.1 IEEET1 参数说明

| 参数 | 典型值 | 说明 |
|------|--------|------|
| KA | 50-400 | 电压调节器增益 |
| TA | 0.01-0.1 s | 调节器时间常数 |
| KE | 1.0 (或 -0.05) | 励磁机常数 |
| TE | 0.5-1.0 s | 励磁机时间常数 |
| KF | 0.01-0.1 | 稳定反馈增益 |
| TF | 0.5-1.5 s | 稳定反馈时间常数 |
| SE1, SE2 | 0.03-0.15 | 饱和系数 |

**饱和函数**: SE(Efd) = A_sat * exp(B_sat * Efd)

In [ ]:
# ===== IEEET1 参数创建 =====
# 典型晶闸管励磁 (Kundur Ch.12)
exc = IEEET1Params(
    KA=200, TA=0.02,
    KE=1.0, TE=0.5,
    KF=0.05, TF=1.0,
    VR_MIN=-1.0, VR_MAX=5.0,
    Efd_MIN=0.0, Efd_MAX=5.0
)

print(f'KA={exc.KA}, TA={exc.TA}, TE={exc.TE}, KF={exc.KF}')
print(f'Saturation: A={exc._A_sat:.6f}, B={exc._B_sat:.4f}')

# 验证饱和函数在工作点
print(f'SE(5.0)={exc.saturation(5.0):.4f} (should = SE1={exc.SE1})')
print(f'SE(3.75)={exc.saturation(3.75):.4f} (should ~ SE2={exc.SE2})')

---
## 8.2 开环阶跃响应

Vref 从 1.0 阶跃到 1.05 pu，观察 Efd 的响应。
开环意味着没有发电机模型，直接看励磁系统本身的动态。

In [ ]:
# ===== 开环阶跃响应 =====
dt = 0.001
t_end = 2.0
n_steps = int(t_end / dt)
t = np.linspace(0, t_end, n_steps + 1)

state = None
efd_history = np.zeros(n_steps + 1)

for i in range(n_steps + 1):
    V_ref = 1.05 if t[i] >= 0.05 else 1.0  # 0.05s 时阶跃
    efd, state = exc.compute(V_ref=V_ref, V_measured=1.0, V_S=0.0,
                              dt=dt, state=state)
    efd_history[i] = efd

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, efd_history, 'b-', linewidth=2)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Initial Efd=1.0')
ax.axvline(x=0.05, color='r', linestyle=':', alpha=0.5, label='Step at t=0.05s')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Efd (pu)')
ax.set_title('IEEET1 Open-Loop Step Response (Vref: 1.0 -> 1.05)')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('examples/kundur/ieeet1_step.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final Efd: {efd_history[-1]:.4f} pu')
print(f'Settling time (approx): steady state reached in ~1s')

---
## 8.3 参数扫描: KA (增益) 的影响

增大 KA 使响应更快，但过高的 KA 可能引起振荡。

In [ ]:
# ===== KA 扫描 =====
KA_vals = [50, 100, 200, 400]
fig, ax = plt.subplots(figsize=(10, 5))

for KA in KA_vals:
    exc_i = IEEET1Params(KA=KA, TA=0.02, TE=0.5)
    state = None
    efd_hist = np.zeros(n_steps + 1)
    for i in range(n_steps + 1):
        V_ref = 1.05 if t[i] >= 0.05 else 1.0
        efd, state = exc_i.compute(V_ref=V_ref, V_measured=1.0, V_S=0.0,
                                    dt=dt, state=state)
        efd_hist[i] = efd
    ax.plot(t, efd_hist, linewidth=1.5, label=f'KA={KA}')

ax.axvline(x=0.05, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Time (s)'); ax.set_ylabel('Efd (pu)')
ax.set_title('Effect of Regulator Gain KA on Step Response')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('examples/kundur/ieeet1_ka_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('KA 越大 -> 响应越快，但可能引入振荡 (K5 < 0 时增加负阻尼)')

---
## 8.4 稳定反馈 KF 的作用

KF 提供励磁电压变化的负反馈 ( derivative feedback )，
改善励磁系统的阻尼特性。

In [ ]:
# ===== KF 扫描 =====
KF_vals = [0.0, 0.02, 0.05, 0.1]
fig, ax = plt.subplots(figsize=(10, 5))

for KF in KF_vals:
    exc_i = IEEET1Params(KA=200, TA=0.02, TE=0.5, KF=KF, TF=1.0)
    state = None
    efd_hist = np.zeros(n_steps + 1)
    for i in range(n_steps + 1):
        V_ref = 1.05 if t[i] >= 0.05 else 1.0
        efd, state = exc_i.compute(V_ref=V_ref, V_measured=1.0, V_S=0.0,
                                    dt=dt, state=state)
        efd_hist[i] = efd
    ax.plot(t, efd_hist, linewidth=1.5, label=f'KF={KF}')

ax.set_xlabel('Time (s)'); ax.set_ylabel('Efd (pu)')
ax.set_title('Effect of Stabilizing Feedback KF on Step Response')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('examples/kundur/ieeet1_kf_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('KF 增大 -> 超调减小，响应变慢 (稳定性 vs 响应速度权衡)')

---
## 8.5 饱和特性

励磁机的饱和特性 SE(Efd) 影响大信号响应。

In [ ]:
# ===== 饱和曲线 =====
Efd_range = np.linspace(0, 5.5, 100)
SE_vals = np.array([exc.saturation(e) for e in Efd_range])

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(Efd_range, SE_vals, 'b-', linewidth=2)
ax.scatter([3.75, 5.0], [exc.SE2, exc.SE1], color='red', s=50, zorder=5)
ax.annotate(f'SE2={exc.SE2}', (3.75, exc.SE2), textcoords='offset points',
            xytext=(10, 10), fontsize=10)
ax.annotate(f'SE1={exc.SE1}', (5.0, exc.SE1), textcoords='offset points',
            xytext=(10, -15), fontsize=10)
ax.set_xlabel('Efd (pu)'); ax.set_ylabel('SE(Efd)')
ax.set_title('Exciter Saturation Characteristic')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('examples/kundur/ieeet1_saturation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8.6 AVR 与发电机闭环响应

结合第12章的 Heffron-Phillips 模型，分析 AVR 对发电机端电压调节
和小干扰稳定的综合影响。

**关键结论 (DeMello-Concordia, 1969):**
1. 高 KA 改善电压调节速度，但通过 K5<0 通道引入负阻尼
2. KF 稳定反馈能部分补偿负阻尼
3. **PSS** 是最有效的解决方案 — 通过 Delta_omega 信号提供纯正阻尼

> 详见 [Ch.12a SMIB 小干扰稳定](./ch12a_smib_small_signal.ipynb) 中的根轨迹和 PSS 设计。

In [ ]:
# ===== 闭环概念演示: 不同 KA 下的特征值变化 =====
from psa4teaching.stability import compute_heffron_phillips_constants

KA_range = np.logspace(0.5, 3, 20)
zeta_em = []
for Ka in KA_range:
    r = compute_heffron_phillips_constants(
        E_prime=1.2, V_infinity=1.0, X_total=0.5,
        Xd=1.8, Xd_prime=0.3, Xq=1.7,
        Td0_prime=8.0, H=5.0, D=0.0,
        delta_0=np.radians(30), Ka=Ka, Te=0.5, verbose=False)
    # 提取机电模式阻尼
    for ev, f in zip(r.eigenvalues, r.frequencies):
        if f > 0.1 and f < 5.0:  # 机电模式
            zeta_em.append(-ev.real / abs(ev))
            break

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(KA_range, zeta_em, 'b-o', markersize=4, linewidth=1.5)
ax.axhline(y=0.05, color='r', linestyle='--', label='Min acceptable (5%)')
ax.axhline(y=0, color='k', linewidth=0.5)
ax.set_xlabel('KA (AVR Gain)'); ax.set_ylabel('Electromechanical Damping Ratio')
ax.set_title('AVR Gain vs Damping (K5 < 0 causing negative damping)')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig('examples/kundur/avr_damping_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'At KA=10: zeta={zeta_em[0]:.4f}')
print(f'At KA=1000: zeta={zeta_em[-1]:.4f} (degraded)')
print('AVR 增益 vs 阻尼: 典型的工程权衡')

---
## 本章小结

1. **IEEET1** 是标准的励磁系统模型，包含调节器 + 励磁机 + 稳定反馈
2. **KA (增益)**: 越大响应越快，但可能降低阻尼
3. **KF (稳定反馈)**: 改善励磁系统本身的稳定性，但不能完全补偿机电模式负阻尼
4. **饱和**: 在大信号下限制励磁输出，符合物理实际
5. **闭环分析**: 需结合发电机模型 (Ch.12 Heffron-Phillips) 才能完整评估 AVR 影响

> Kundur 参考: Section 8.6 — Excitation System Stabilizing Circuits
> DeMello F.P., Concordia C. "Concepts of Synchronous Machine Stability as Affected by Excitation Control" IEEE Trans. PAS, 1969